In [27]:
from scipy.special import softmax
import torch
import numpy as np
import torch.nn as nn
from transformers import AutoConfig, AutoModelForSequenceClassification
from safetensors.torch import load_file  # comes with HF if safetensors installed
import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import pandas as pd
import os
from captum.attr import IntegratedGradients
import torch

In [28]:
ckpt_path = "projects/nace_classification/nace_report_topic_analysis/results/BERT_models/results__new_approach_data__num_layers_2bert-base-uncased__train_full_model__some_labels/checkpoint-15990"

In [29]:
# load checkpoint
config = AutoConfig.from_pretrained(ckpt_path)

# 2. Build a model from config (bare BertForSequenceClassification)
model = AutoModelForSequenceClassification.from_config(config)

# 3. Rebuild the SAME classifier architecture as in training
hidden = getattr(config, "custom_hidden", 512)  # fallback if not in config
num_layers = getattr(config, "custom_num_layers", 2)

layers = []
for i in range(num_layers):
    in_dim = config.hidden_size if i == 0 else hidden
    layers.append(nn.Linear(in_dim, hidden))
    layers.append(nn.GELU())
    layers.append(nn.Dropout(0.2))
layers.append(nn.Linear(hidden, config.num_labels))
model.classifier = nn.Sequential(*layers)

# 4. Load weights from model.safetensors
state_dict = load_file(os.path.join(ckpt_path, "model.safetensors"))
model.load_state_dict(state_dict, strict=True)  # will fail loudly if mismatch

# 5. Inference mode
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [30]:
tokenizer = AutoTokenizer.from_pretrained(ckpt_path) 

In [31]:
chunk = "registered debt securities debentures and loans as well as other loans are carried at acquisition cost taking into account amortisation or at the lower fair value."

In [32]:
inputs = tokenizer(chunk, return_tensors="pt", truncation=True)
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits.squeeze(0)
    logits = softmax(logits)

    label_scores = {
        model.config.id2label[i]: logits[i].item()
        for i in range(len(logits))
    }
    label_scores = dict(sorted(label_scores.items(), key=lambda x: x[1], reverse=True))


In [33]:
label_scores

{'K': 0.8631665110588074,
 'NO_CLASS': 0.12547606229782104,
 'L': 0.003961633890867233,
 'N': 0.0025254436768591404,
 'H': 0.0011446777498349547,
 'G': 0.0009637715411372483,
 'I': 0.000567516777664423,
 'M': 0.0005168082425370812,
 'F': 0.0003902418538928032,
 'D': 0.00031932300771586597,
 'E': 0.00026744353817775846,
 'Q': 0.00025465767248533666,
 'B': 0.00017162806761916727,
 'A': 0.00015716318739578128,
 'P': 7.568336150143296e-05,
 'J': 2.3999686163733713e-05,
 'C': 1.7250556993531063e-05}

### Now with captum

In [34]:
predicted_class = list(label_scores.items())[0][0]
predicted_class_id = int(np.argmax(logits))
predicted_class_id

14

In [35]:
np.array(inputs["input_ids"])

array([[  101,  5068,  7016, 12012,  2139, 10609, 22662,  1998, 10940,
         2004,  2092,  2004,  2060, 10940,  2024,  3344,  2012,  7654,
         3465,  2635,  2046,  4070, 16095,  7315,  3370,  2030,  2012,
         1996,  2896,  4189,  3643,  1012,   102]])

In [36]:
inputs = tokenizer(chunk, return_tensors="pt")
input_ids = inputs["input_ids"]          # LONG!
attention_mask = inputs["attention_mask"]

def forward_func(input_ids, attention_mask):
    return model(
        input_ids=input_ids,
        attention_mask=attention_mask
    ).logits

# predicted class
pred = forward_func(input_ids, attention_mask).argmax(dim=1).item()

ig = IntegratedGradients(forward_func)
attr = ig.attribute(
    input_ids,
    target=pred,
    additional_forward_args=(attention_mask,)
)

RuntimeError: Expected tensor for argument #1 'indices' to have one of the following scalar types: Long, Int; but got torch.FloatTensor instead (while checking arguments for embedding)

In [ ]:

model.eval()
ig = IntegratedGradients(model)

inputs = tokenizer(chunk, return_tensors="pt")
outputs = model(**inputs)

attributions = ig.attribute(
    
    inputs["input_ids"].long(),
    target=predicted_class_id,
    additional_forward_args=(inputs["attention_mask"],)
)

RuntimeError: Expected tensor for argument #1 'indices' to have one of the following scalar types: Long, Int; but got torch.FloatTensor instead (while checking arguments for embedding)

In [ ]:
type(inputs["input_ids"]), type(predicted_class_id), type(inputs["attention_mask"])

(torch.Tensor, int)